# Importing the libaries

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score


# Pandas


## mounting the data
- mounting the dataset
- making copy of the data
- viwing the first 5 rows

In [11]:
path = "/content/drive/MyDrive/dataset/data.csv"

df_original = pd.read_csv(path)
df = df_original.copy()
print("the original data", df_original.shape)
print("the working data", df.shape)
df.head()

the original data (29332, 87)
the working data (29332, 87)


,android.permission.GET_ACCOUNTS,com.sonyericsson.home.permission.BROADCAST_BADGE,android.permission.READ_PROFILE,android.permission.MANAGE_ACCOUNTS,android.permission.WRITE_SYNC_SETTINGS,android.permission.READ_EXTERNAL_STORAGE,android.permission.RECEIVE_SMS,com.android.launcher.permission.READ_SETTINGS,android.permission.WRITE_SETTINGS,com.google.android.providers.gsf.permission.READ_GSERVICES,...,com.android.launcher.permission.UNINSTALL_SHORTCUT,com.sec.android.iap.permission.BILLING,com.htc.launcher.permission.UPDATE_SHORTCUT,com.sec.android.provider.badge.permission.WRITE,android.permission.ACCESS_NETWORK_STATE,com.google.android.finsky.permission.BIND_GET_INSTALL_REFERRER_SERVICE,com.huawei.android.launcher.permission.READ_SETTINGS,android.permission.READ_SMS,android.permission.PROCESS_INCOMING_CALLS,Result
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0


## Data Info
- the shape of the data

In [12]:
print("number of row and colums", df.shape)
#print("\n", "#" * 50 , "\n")
#df.info()
#print("\n", "#" * 50 , "\n")
#print(df.columns)
#print("\n", "#" * 50 , "\n")
#print(df.describe())
#print("\n", "#" * 50 , "\n")
#df['Result'].value_counts()



number of row and colums (29332, 87)


## Data Cleaning
- droping null values
- removing duplicates values


In [13]:
print("the number of null values in the dataset",df_original.isnull().sum().sum())
df.dropna(inplace=True)
print("the number of duplicates in the dataset", df_original.duplicated().sum())
df.drop_duplicates(inplace=True)

df.isnull().sum().sort_values(ascending=False).head(10)


the number of null values in the dataset 0
the number of duplicates in the dataset 21841


,0
android.permission.GET_ACCOUNTS,0
com.sonyericsson.home.permission.BROADCAST_BADGE,0
android.permission.READ_PROFILE,0
android.permission.MANAGE_ACCOUNTS,0
android.permission.WRITE_SYNC_SETTINGS,0
android.permission.READ_EXTERNAL_STORAGE,0
android.permission.RECEIVE_SMS,0
com.android.launcher.permission.READ_SETTINGS,0
android.permission.WRITE_SETTINGS,0
com.google.android.providers.gsf.permission.READ_GSERVICES,0


In [14]:
print("the original data", df_original.shape)
print("the working data", df.shape)

the original data (29332, 87)
the working data (7491, 87)


In [15]:
print(df['Result'].value_counts())
print("#" * 50)
df.dtypes.value_counts()

Result
0    4867
1    2624
Name: count, dtype: int64
##################################################


,count
int64,87


# **Scikit learn**
- spliting the data into train and split:
x = permissions and y = Result
- why the split ? the modul analyze the permissions (X) and learn what to expect (y) to be, so when the app sent a list of an app permissions the modul will have the ability of predict wether the app is begine or malware
- Y = series in python meaning one column and multiable rows

In [16]:
X = df.drop(columns=['Result'])
y = df['Result']

print(X.shape)
print(y.shape)
print(y.head())


(7491, 86)
(7491,)
0    0
1    0
2    0
3    0
4    0
Name: Result, dtype: int64


## **train_test_split**
- Training set: نستخدمها لتدريب النموذج
- Testing set: نستخدمها لتقييم أداء النموذج على بيانات ما شافها قبلًا، حتى نتأكد أن النموذج ما تعلم حفظ البيانات (Overfitting).

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(5992, 86)
(1499, 86)
(5992,)
(1499,)


# **The model object**

## RF 1
* **100 trees** → Good accuracy/speed balance; performance plateaus after this.
* **max_depth=None** → Deep trees capture complex, non-linear malware patterns.
* **n_jobs=-1** → Uses all CPU cores; trees train in parallel.

In [18]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(y_pred_rf[:10])
print(y_test.values[:10])

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[0 0 0 1 1 0 0 1 0 0]
[0 0 0 1 1 0 0 1 0 0]
[[970  45]
 [ 47 437]]
              precision    recall  f1-score   support

           0       0.95      0.96      0.95      1015
           1       0.91      0.90      0.90       484

    accuracy                           0.94      1499
   macro avg       0.93      0.93      0.93      1499
weighted avg       0.94      0.94      0.94      1499



In [19]:
y_proba = rf.predict_proba(X_test)[:, 1]
threshold = 0.3
y_pred_custom = (y_proba >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))


[[906 109]
 [ 27 457]]
              precision    recall  f1-score   support

           0       0.97      0.89      0.93      1015
           1       0.81      0.94      0.87       484

    accuracy                           0.91      1499
   macro avg       0.89      0.92      0.90      1499
weighted avg       0.92      0.91      0.91      1499



## RF 2 after tuning
* **600 trees** →
* **max_depth=3** →
* **n_jobs=-1** → Uses all CPU cores; trees train in parallel.

In [20]:
rf_tuned = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=3,
    max_features='sqrt',
    # class_weight={0:1,1:2},
    class_weight={0:1, 1:5},
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)

y_pred_tuned = rf_tuned.predict(X_test)


print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))


[[919  96]
 [ 22 462]]
              precision    recall  f1-score   support

           0       0.98      0.91      0.94      1015
           1       0.83      0.95      0.89       484

    accuracy                           0.92      1499
   macro avg       0.90      0.93      0.91      1499
weighted avg       0.93      0.92      0.92      1499



In [21]:
y_prob_new = rf_tuned.predict_proba(X_test)[:,1]

threshold = 0.3
y_pred_custom = (y_prob_new >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[833 182]
 [ 10 474]]
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1015
           1       0.72      0.98      0.83       484

    accuracy                           0.87      1499
   macro avg       0.86      0.90      0.86      1499
weighted avg       0.90      0.87      0.88      1499



In [22]:
print(df.columns.tolist())
print("\nNew features exist?")
print(all(col in df.columns for col in [
    'number_of_permissions',
    'sensitive_permission_count',
    'network_permission_ratio',
    'persistence_permission_count',
    'low_perm_high_net',
    'stealth_data_access',
    'minimal_malware_pattern']))

['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES', 'android.permission.DOWNLOAD_WITHOUT_NOTIFICATION', 'android.permission.GET_TASKS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'android.permission.RECORD_AUDIO', 'com.huawei.android.launcher.permission.CHANGE_BADGE', 'com.oppo.launcher.permission.READ_SETTINGS', 'android.permission.CHANGE_NETWORK_STATE', 'com.android.launcher.permission.INSTALL_SHORTCUT', 'android.permission.android.permission.READ_PHONE_STATE', 'android.permission.CALL_PHONE', 'android.permission.WRITE_CONTACTS', 'android.permission.READ_PHONE_STATE', 'com.samsung.android.providers.context.permi

In [23]:
importances = rf_tuned.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print(feat_imp.head(20))
print(feat_imp.tail(20))


android.permission.READ_PHONE_STATE                                       0.271567
com.google.android.c2dm.permission.RECEIVE                                0.209177
com.android.launcher.permission.INSTALL_SHORTCUT                          0.061786
android.permission.READ_EXTERNAL_STORAGE                                  0.043866
com.android.vending.BILLING                                               0.041414
android.permission.RECEIVE_BOOT_COMPLETED                                 0.031777
android.permission.SYSTEM_ALERT_WINDOW                                    0.021980
android.permission.ACCESS_COARSE_LOCATION                                 0.021219
com.google.android.providers.gsf.permission.READ_GSERVICES                0.020490
android.permission.GET_TASKS                                              0.017727
android.permission.CAMERA                                                 0.015978
android.permission.ACCESS_WIFI_STATE                                      0.014314
com.

In [24]:
low_feats = feat_imp[feat_imp < 0.001].index

X_train_red = X_train.drop(columns=low_feats)
X_test_red  = X_test.drop(columns=low_feats)

rf_tuned.fit(X_train_red, y_train)


RandomForestClassifier(class_weight={0: 1, 1: 5}, min_samples_leaf=3,
                       n_estimators=600, n_jobs=-1, random_state=42)

# saving the model

In [25]:
final_model = {
    "model": rf_tuned,
    "threshold": 0.3,
    "features": X.columns.tolist()
}

with open("malware_rf_final.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Final model saved successfully ✅")


Final model saved successfully ✅


In [30]:
with open("malware_rf_final.pkl", "rb") as f:
    model_test = pickle.load(f)

# NumPy Function




##  help us to learn what is the correct order of the conftion matrixes
  [[TN  FP]

  [FN  TP]]

In [27]:
def extract_confusion_elements(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    return TP, FN, FP, TN

TP, FN, FP, TN = extract_confusion_elements(y_test, y_pred_rf)

print("TP:", TP)
print("FN:", FN)
print("FP:", FP)
print("TN:", TN)

recall = TP / (TP + FN)
precision = TP / (TP + FP)

print("Recall:", recall)
print("Precision:", precision)



TP: 437
FN: 47
FP: 45
TN: 970
Recall: 0.9028925619834711
Precision: 0.9066390041493776
